# OGDC leakage case — the bug tsauditor was built to catch

A direction-prediction model on Pakistani equity (OGDC) data reached ~99.7%
accuracy — almost entirely because a same-day percentage-change feature
(`ChangeP`) was mathematically near-identical to the `Direction` target it was
meant to predict. This notebook shows `tsauditor` catching that leak directly
from the data, and why the rank/AUC test succeeds where a Pearson threshold
would have missed it.

The full model comparison (accuracy with vs. without the leak) lives in
`compare_leakage.py` in this folder; here we focus on detection.

In [1]:
import pandas as pd
import tsauditor as tsa

df = pd.read_csv("ogdc_with_regimes.csv", index_col="Date", parse_dates=True)
df = df.dropna(subset=["Direction"])
print("rows:", len(df), "| columns:", len(df.columns))
print("target: Direction (binary up/down)")
df[["Price", "ChangeP", "Returns", "Direction"]].head()

rows: 1537 | columns: 24
target: Direction (binary up/down)


,Price,ChangeP,Returns,Direction
Date,,,,
2020-01-22,143.89,-3.37,-3.371164,0
2020-01-23,143.05,-0.58,-0.583779,0
2020-01-24,143.71,0.46,0.461377,1
2020-01-27,140.85,-1.99,-1.990119,0
2020-01-28,140.07,-0.55,-0.553781,0


## Audit for leakage

One call. `leaky_columns()` is the shortlist of features to review or remove
before training.

In [2]:
report = tsa.scan(df, target="Direction", domain="finance", run_stationarity=False)

print("critical:", len(report.critical), "| warnings:", len(report.warnings))
print("leaky_columns:", report.leaky_columns())

critical: 2 | warnings: 39
leaky_columns: ['ChangeP', 'Returns']


`ChangeP` and `Returns` are both **same-day** quantities whose sign *defines*
`Direction`, so they reproduce the target. tsauditor flags them **LEK001**
(CRITICAL, target equivalence):

In [3]:
for i in report.filter(code="LEK001"):
    print(
        i.column,
        "| metric:",
        i.evidence["metric"],
        "| separation:",
        i.evidence["separation"],
    )
    print("   suggestion:", i.suggestion)
    print()

ChangeP | metric: auc | separation: 1.0
   suggestion: Remove or reconstruct column 'ChangeP': it near-deterministically reproduces the target variable and will leak. Keep it only if you can confirm it is genuinely available at prediction time.

Returns | metric: auc | separation: 1.0
   suggestion: Remove or reconstruct column 'Returns': it near-deterministically reproduces the target variable and will leak. Keep it only if you can confirm it is genuinely available at prediction time.



## Why Pearson would have missed it

`Direction` is binary (0/1). Pearson correlation against a binary target is
*point-biserial*, which is capped near `sqrt(2/pi) ~ 0.798` — it can never reach
1.0 no matter how perfectly the feature defines the target's sign. So a
correlation-threshold detector under-rates the leak (below you'll see `ChangeP`
score well under the ceiling), while AUC separation scores it 1.0.

In [4]:
pearson = abs(df["ChangeP"].corr(df["Direction"].astype(float)))
auc_sep = next(
    i for i in report.filter(code="LEK001") if i.column == "ChangeP"
).evidence["separation"]
print(
    f"ChangeP vs Direction  Pearson |r| = {pearson:.3f}  (point-biserial ceiling ~0.798)"
)
print(f"ChangeP vs Direction  AUC sep     = {auc_sep:.3f}  (catches it cleanly)")

ChangeP vs Direction  Pearson |r| = 0.627  (point-biserial ceiling ~0.798)
ChangeP vs Direction  AUC sep     = 1.000  (catches it cleanly)


## The measured impact

Removing the leaked features (`ChangeP`, `Returns`, and the same-day `Open`/
`High`/`Low`, which are equally unavailable at prediction time) collapses the
headline accuracy — the number was mostly an artifact of the leak:

| Model | With leakage | Without leakage |
|-------|--------------|-----------------|
| Random Forest | 99.68% (AUC 0.9987) | 69.81% (AUC 0.7795) |
| Gradient Boosting | 99.68% (AUC 0.9967) | 73.70% (AUC 0.8072) |

Run `python compare_leakage.py` in this folder to reproduce the full experiment
(requires scikit-learn).

**Takeaway:** a single `tsa.scan(...)` before training surfaces `leaky_columns()`
so this class of mistake never reaches the model.